# Następny eksperyment po UnCLIP: EEG → embedding → retrieval/reranking

Ten notebook jest kontynuacją po pełnym wyniku `unclip_mole_generation_full`.

Wynik, który mamy w folderze `wyniki colab`, mówi jasno:

- Stable UnCLIP z EEG: `SSIM ≈ 0.173`, `PSNR ≈ 9.33`, `L1 ≈ 0.292`,
- Stable UnCLIP oracle: `SSIM ≈ 0.246`, `PSNR ≈ 9.59`, `L1 ≈ 0.279`,
- lokalny VAE ensemble dla `mole`: `SSIM ≈ 0.286`, `PSNR ≈ 11.60`, `L1 ≈ 0.236`.

Czyli sam pretrained UnCLIP nie wystarcza: nawet oracle jest poniżej VAE. Gridy pokazują też, że EEG→CLIP często ląduje w złej kategorii. Następny sensowny ruch to sprawdzić, czy ten sam sygnał EEG lepiej działa jako **retrieval/reranking**: wybieramy najbliższy obraz/kandydata w przestrzeni embeddingów zamiast prosić dyfuzję o pełną rekonstrukcję z rozmytego wektora.

## Co ten notebook zrobi

1. Montuje Drive i rozpakowuje te same paczki co notebook UnCLIP.
2. Odtwarza albo wczytuje embeddingi CLIP/UnCLIP obrazów.
3. Wczytuje wytrenowany model `unclip_mole_retrieval/eeg_image_retrieval.pt`.
4. Dla każdego testowego obrazu uśrednia embeddingi EEG z powtórzeń.
5. Wybiera najbliższy obraz kandydujący w przestrzeni embeddingów.
6. Liczy top-1/top-5/category accuracy oraz L1/PSNR/SSIM obrazu wybranego przez retrieval.
7. Porównuje: VAE vs UnCLIP EEG/oracle vs retrieval nearest-neighbor.

To nie jest jeszcze finał rekonstrukcji. To test hipotezy: czy przed kolejną dyfuzją warto dodać etap wyboru/rerankingu kandydatów.

In [ ]:
# Konfiguracja eksperymentu
from pathlib import Path

PARTICIPANT = 'mole'

DRIVE_DATA = Path('/content/drive/MyDrive/biai/data')
DRIVE_RESULTS = Path('/content/drive/MyDrive/biai/results')

EPOCHS_ZIP = DRIVE_DATA / 'biai_eeg_qc_0_0p8.zip'
ASSETS_ZIP = DRIVE_DATA / 'biai_unclip_assets.zip'

ROOT = Path('/content/biai_retrieval_reranking')
MANIFEST_DIR = ROOT / f'reconstruction_manifests/participant_image_{PARTICIPANT}_no_abc'
EMBEDDING_DIR = ROOT / f'image_embeddings_unclip_participant_image_{PARTICIPANT}_no_abc'

RETRIEVAL_DIR = DRIVE_RESULTS / f'unclip_{PARTICIPANT}_retrieval'
UNCLIP_FULL_DIR = DRIVE_RESULTS / f'unclip_{PARTICIPANT}_generation_full'
OUTPUT_DIR = DRIVE_RESULTS / f'retrieval_reranking_{PARTICIPANT}_unclip_embedding'

# Branch, z którego notebook może awaryjnie pobrać brakujący skrypt.
GITHUB_BRANCH = 'codex-eeg-pipeline-qc'

# Fallback z lokalnego sweepu VAE z 2026-06-26.
VAE_BASELINE = {
    'model': 'VAE ensemble (mole)',
    'images': 44,
    'l1': 0.2355921593579379,
    'psnr': 11.59615940397436,
    'ssim': 0.2864757523956624,
}

print('ROOT =', ROOT)
print('RETRIEVAL_DIR =', RETRIEVAL_DIR)
print('UNCLIP_FULL_DIR =', UNCLIP_FULL_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

In [ ]:
# Montowanie Drive i instalacja zależności
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import subprocess
import sys

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '-q',
    'transformers', 'safetensors', 'scikit-learn'
], check=True)

print('Drive zamontowany i zależności gotowe.')

In [ ]:
# Rozpakowanie danych i assets. Ten krok można bezpiecznie powtarzać.
import shutil
import zipfile

def resolve_drive_file(path):
    path = Path(path)
    if path.is_file():
        return path
    drive_root = Path('/content/drive/MyDrive')
    matches = sorted(drive_root.rglob(path.name)) if drive_root.exists() else []
    if len(matches) == 1:
        print(f'Znalazłem {path.name} pod inną ścieżką:', matches[0])
        return matches[0]
    raise FileNotFoundError(f'Nie znalazłem {path}. Dopilnuj, żeby ZIP był na Drive w MyDrive/biai/data/.')

if ROOT.exists():
    shutil.rmtree(ROOT)
ROOT.mkdir(parents=True)

for source in map(resolve_drive_file, [EPOCHS_ZIP, ASSETS_ZIP]):
    print('Rozpakowuję:', source)
    with zipfile.ZipFile(source) as archive:
        archive.extractall(ROOT)

print('Pliki w ROOT:', sorted(p.name for p in ROOT.iterdir()))

In [ ]:
# Starsza paczka biai_unclip_assets.zip mogła nie zawierać reconstruct_nearest_neighbor.py.
# Jeśli go brakuje, pobieramy świeży skrypt z GitHuba.
from urllib.request import urlretrieve

scripts_dir = ROOT / 'scripts'
scripts_dir.mkdir(exist_ok=True)
script_path = scripts_dir / 'reconstruct_nearest_neighbor.py'

if not script_path.is_file():
    raw_url = f'https://raw.githubusercontent.com/taf4you2/biai/{GITHUB_BRANCH}/scripts/reconstruct_nearest_neighbor.py'
    print('Brak reconstruct_nearest_neighbor.py w assets; pobieram:', raw_url)
    urlretrieve(raw_url, script_path)

required_scripts = [
    scripts_dir / 'train_eeg_image_retrieval.py',
    scripts_dir / 'extract_unclip_image_embeddings.py',
    scripts_dir / 'reconstruct_nearest_neighbor.py',
]
missing = [str(path) for path in required_scripts if not path.is_file()]
if missing:
    raise FileNotFoundError('Brak wymaganych skryptów: ' + ', '.join(missing))
print('Skrypty gotowe.')

In [ ]:
# Sprawdzenie, czy pełny UnCLIP i retrieval istnieją na Drive.
import json

for required in [RETRIEVAL_DIR / 'eeg_image_retrieval.pt', UNCLIP_FULL_DIR / 'unclip_generation_summary.json']:
    if not required.is_file():
        raise FileNotFoundError(f'Brak wymaganego wyniku: {required}')

unclip_summary = json.loads((UNCLIP_FULL_DIR / 'unclip_generation_summary.json').read_text(encoding='utf-8'))
print(json.dumps(unclip_summary, indent=2, ensure_ascii=False))

In [ ]:
# Odtworzenie embeddingów obrazów, jeśli nie ma ich jeszcze w świeżym runtime.
def run_logged(command, log_path):
    log_path = Path(log_path)
    log_path.parent.mkdir(parents=True, exist_ok=True)
    print('RUN:', ' '.join(map(str, command)))
    with log_path.open('a', encoding='utf-8') as log:
        process = subprocess.Popen(command, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in process.stdout:
            print(line, end='')
            log.write(line)
            log.flush()
        if process.wait() != 0:
            raise RuntimeError('Polecenie nie powiodło się: ' + ' '.join(map(str, command)))

if not (EMBEDDING_DIR / 'image_embeddings.npy').is_file():
    run_logged([
        sys.executable, '-u', 'scripts/extract_unclip_image_embeddings.py',
        '--manifest-dir', str(MANIFEST_DIR),
        '--project-root', str(ROOT),
        '--output-dir', str(EMBEDDING_DIR),
        '--batch-size', '8',
    ], DRIVE_RESULTS / f'retrieval_reranking_{PARTICIPANT}_extract_embeddings.log')
else:
    print('Embeddingi już istnieją:', EMBEDDING_DIR)

In [ ]:
# Właściwy eksperyment: EEG embedding -> nearest neighbor/reranking.
summary_path = OUTPUT_DIR / 'reconstruction_summary.json'

if summary_path.is_file():
    print('Retrieval/reranking już gotowy:', summary_path)
elif OUTPUT_DIR.exists():
    raise RuntimeError(f'Katalog istnieje, ale jest niekompletny: {OUTPUT_DIR}. Usuń go albo zmień OUTPUT_DIR.')
else:
    run_logged([
        sys.executable, '-u', 'scripts/reconstruct_nearest_neighbor.py',
        '--result-dir', str(RETRIEVAL_DIR),
        '--project-root', str(ROOT),
        '--output-dir', str(OUTPUT_DIR),
        '--batch-size', '256',
        '--grid-rows', '8',
        '--image-size', '220',
        '--metric-size', '256',
    ], DRIVE_RESULTS / f'retrieval_reranking_{PARTICIPANT}.log')

nn_summary = json.loads(summary_path.read_text(encoding='utf-8'))
print(json.dumps(nn_summary, indent=2, ensure_ascii=False))

In [ ]:
# Porównanie metryk: VAE vs UnCLIP vs retrieval/reranking.
import pandas as pd
from IPython.display import display

rows = [VAE_BASELINE]
rows.append({
    'model': 'Stable UnCLIP EEG full',
    'images': unclip_summary['images'],
    'l1': unclip_summary['eeg']['l1'],
    'psnr': unclip_summary['eeg']['psnr'],
    'ssim': unclip_summary['eeg']['ssim'],
})
rows.append({
    'model': 'Stable UnCLIP oracle full',
    'images': unclip_summary['images'],
    'l1': unclip_summary['oracle']['l1'],
    'psnr': unclip_summary['oracle']['psnr'],
    'ssim': unclip_summary['oracle']['ssim'],
})
nn_image_metrics = nn_summary.get('image_averaged_image_metrics', {})
rows.append({
    'model': 'EEG nearest-neighbor image-averaged',
    'images': nn_summary['image_averaged']['samples'],
    'l1': nn_image_metrics.get('l1'),
    'psnr': nn_image_metrics.get('psnr'),
    'ssim': nn_image_metrics.get('ssim'),
})

comparison = pd.DataFrame(rows)
display(comparison.sort_values('ssim', ascending=False).reset_index(drop=True))

retrieval_table = pd.DataFrame([
    {'level': 'sample', **nn_summary['sample_level']},
    {'level': 'image_averaged', **nn_summary['image_averaged']},
])
display(retrieval_table)

In [ ]:
# Podgląd gridów retrieval/reranking.
from IPython.display import Image, Markdown

grid_dir = OUTPUT_DIR / 'grids'
for grid in [
    grid_dir / 'best_reconstructions.jpg',
    grid_dir / 'category_correct_reconstructions.jpg',
    grid_dir / 'worst_reconstructions.jpg',
]:
    if grid.is_file():
        display(Markdown(f'### {grid.name}'))
        display(Image(filename=str(grid)))
    else:
        print('Brak gridu:', grid)

In [ ]:
# Automatyczny werdykt i następny ruch.
from IPython.display import Markdown

best_row = comparison.sort_values('ssim', ascending=False).iloc[0]
nn_top1 = nn_summary['image_averaged']['top1']
nn_top5 = nn_summary['image_averaged']['top5']
nn_cat = nn_summary['image_averaged']['category_top1']

message = []
message.append(f"Najlepszy model po SSIM: **{best_row['model']}** (`SSIM={best_row['ssim']:.3f}`).")
message.append(f"Retrieval po uśrednieniu powtórzeń: top-1 `{nn_top1:.2%}`, top-5 `{nn_top5:.2%}`, zgodność kategorii `{nn_cat:.2%}`.")

if best_row['model'].startswith('EEG nearest-neighbor'):
    message.append('Wniosek: EEG lepiej działa jako wybór/reranking kandydatów niż jako bezpośredni warunek dla Stable UnCLIP. Następny notebook powinien robić candidate-constrained diffusion albo category-aware reranking.')
elif best_row['model'].startswith('VAE'):
    message.append('Wniosek: VAE nadal jest najmocniejszym punktem odniesienia. Następny krok to poprawa dekodera EEG→embedding przed kolejną próbą generatora.')
else:
    message.append('Wniosek: generator ma przewagę, ale trzeba sprawdzić, czy retrieval może poprawić warunkowanie i ograniczyć pomyłki kategorii.')

display(Markdown('\n\n'.join(message)))